# Water Rights Restored to the Gila River

The impacts of irrigation on vegetation health in the Gila River Valley

## Set up

To get started on this notebook, you’ll need to restore any variables from previous notebooks to your workspace. To save time and memory, make sure to specify which variables you want to load.

In [1]:
### load the project folder and the Gila River boundary from the previous notebook
%store -r project your_gdf

You will also need to import any libraries you are using in this
notebook, since they won’t carry over from the previous notebook:

In [2]:
### import libraries

### working with dates
import pandas as pd
### opening raster files (GeoTIFFs)
import rioxarray as rxr
### combining rasters into one time series
import xarray as xr

## Step 3: Wrangle the raster data

### Step 3A: Load NDVI data

When you ran the code `project.get_data()` in the previous notebook, you actually downloaded the NDVI data in addition to the Gila River Indian Community boundary data. Now you need to load all these downloaded NDVI files into Python. Let’s start by getting all the file names. You will also need to extract the date from the filename. Check out [the lesson on getting information from filenames from EarthDataScience.org](https://www.earthdatascience.org/courses/intro-to-earth-data-science/write-efficient-python-code/loops/data-workflows-with-loops/).

In this lesson, you will use the `glob.glob()` function in Python to find all files that match a **pattern** formed with the **wildcard** character `*`. The wildcard can represent any string of alphanumberic characters. For example, the pattern `'file_*.tif'` will match the files `'file_1.tif'`, `'file_2.tif'`, or even `'file_qeoiurghtfoqaegbn34pf.tif'`, but it will not match `'something-else.csv'` or even `'something-else.tif'`.

In this notebook, we’ll use the `.rglob()`, or **recursive** glob method of the Path object instead. It works similarly, but you’ll notice that we have to convert the results to a list with the `list()` function.

**Watch out!**: `glob` doesn’t necessarily find files in the order you would expect. Make sure to **sort** your file names.

### Reflect and Respond #5:
Take a look at the file names for the NDVI files. If you need a refresher on where earthpy downloaded the files to, you can make a new code chunk and run `print(project.project_dir)` again. What do you notice is the same for all the files? Keep in mind that you only want the NDVI files, not the Quality files (which will flag potential incorrect measurements).

All the NDVI files start with the same text, `MOD13Q1.061__250m_16_days_NDVI_doy`, and end with `_aid0001.tif`. The only part that changes is the date in the middle, for example `2001145`, which means the 145th day of 2001. The Quality files also start with `MOD13Q1.061__250m_16_days_`, but they have `VI_Quality` instead of `NDVI`. So I can use the word `NDVI` to select only the NDVI files.

- Create a **pattern** for the files you want to import. Your pattern should include the parts of the file names that are the same for all files, and replace the rest with the `*` character. Make sure to match the NDVI files, but not the Quality files!  
- Replace `ndvi-pattern` with your pattern  
- Run the code and make sure that you are getting all the files you want and none of the files you don’t

In [3]:
### find all NDVI .tif files (not the Quality files) and sort them by date
ndvi_paths = sorted(list(project.project_dir.rglob('*NDVI*.tif')))

### show the first three and last three file paths to check the pattern
ndvi_paths[:3], ndvi_paths[-3:]

([WindowsPath('C:/Users/chwj1/AppData/Local/earth-analytics/earth-analytics/gila-river-vegetation/gila-river-ndvi/MOD13Q1.061_2001137_to_2022244/MOD13Q1.061__250m_16_days_NDVI_doy2001145000000_aid0001.tif'),
  WindowsPath('C:/Users/chwj1/AppData/Local/earth-analytics/earth-analytics/gila-river-vegetation/gila-river-ndvi/MOD13Q1.061_2001137_to_2022244/MOD13Q1.061__250m_16_days_NDVI_doy2001161000000_aid0001.tif'),
  WindowsPath('C:/Users/chwj1/AppData/Local/earth-analytics/earth-analytics/gila-river-vegetation/gila-river-ndvi/MOD13Q1.061_2001137_to_2022244/MOD13Q1.061__250m_16_days_NDVI_doy2001177000000_aid0001.tif')],
 [WindowsPath('C:/Users/chwj1/AppData/Local/earth-analytics/earth-analytics/gila-river-vegetation/gila-river-ndvi/MOD13Q1.061_2001137_to_2022244/MOD13Q1.061__250m_16_days_NDVI_doy2022209000000_aid0001.tif'),
  WindowsPath('C:/Users/chwj1/AppData/Local/earth-analytics/earth-analytics/gila-river-vegetation/gila-river-ndvi/MOD13Q1.061_2001137_to_2022244/MOD13Q1.061__250m_16_d

### Step 3B: Iterate over all the files to prep them for the analysis 
Now you should have a few dozen files! For each file, you need to:

-   Load the file in using the `rioxarray` library
-   Get the date from the file name
-   Add the date as a dimension coordinate
-   Give your data variable a name

You don’t want to write out the code for each file! That’s a recipe for copy-and-paste issues. Luckily, Python has tools for doing similar tasks repeatedly. In this case, you’ll use one called a `for` loop.

There’s some code below that uses a `for` loop in what is called an **accumulation pattern** to process each file. That means that you will save the results of your processing to a list each time you process the files, and then merge all the arrays in the list.


- Look at the file names. How many characters from the end is the date? `doy_start` and `doy_end` are used to extract the day of the year (doy) from the file name. You will need to count characters from the end and change the values to get the right part of the file name. HINT: the index -1 in Python means the last value, -2 second-to-last, and so on.  
- Replace any required variable names with your chosen variable names

In [5]:
### set the start and end day of year 
### the date (like 2001145) is 25 to 18 characters from the end of the file name
doy_start = -25
doy_end = -18

### empty list to collect each NDVI image
ndvi_das = []

### go through the NDVI files one by one
for ndvi_path in ndvi_paths:

    ### cut the year and day of year out of the file name
    doy = ndvi_path.name[doy_start:doy_end]

    ### change it into a real date (%Y = year, %j = day of year)
    date = pd.to_datetime(doy, format = '%Y%j')

    ### open the tif file, change no-data values to NaN, and remove the band dimension
    da = rxr.open_rasterio(ndvi_path, mask_and_scale = True).squeeze()

    ### add the date to the image
    da = da.assign_coords({'date': date})

    ### make date a dimension so the images can be stacked by date later
    da = da.expand_dims({'date': 1})

    ### call the values 'NDVI'
    da.name = 'NDVI'

    ### add this image to the list
    ndvi_das.append(da)
    

### Step 3C: Combine rasters

Next, stack your arrays by date into a time series using the `xr.combine_by_coords()` function. You will have to tell it which dimension you want to stack your data in.

In [ ]:
### combine NDVI images from all dates

### check it out


## Wrap up Step 3
Don’t forget to store your variables so you can use them in other notebooks! Replace `var1` and `var2` with the variable you want to save, separated by spaces.

In [9]:
%store var1 var2

Finally, be sure to `Restart` and `Run all` to make sure your notebook
works all the way through!